# PriceLens Phase 3A: Structured Text Features

This notebook keeps the original TF-IDF baseline separate and explores structured signals extracted from mixed `catalog_content` text.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BACKEND = ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

ROOT

In [ ]:
import numpy as np
import pandas as pd

from src.data.clean_data import clean_train_data
from src.data.load_data import load_train_data
from src.features.structured_features import CatalogStructuredFeatures, extract_catalog_features
from src.models.train_enhanced_baseline import train_enhanced_baseline

raw = load_train_data()
clean = clean_train_data(raw)
clean.shape

In [ ]:
transformer = CatalogStructuredFeatures()
feature_matrix = transformer.fit_transform(clean["catalog_content"].head(10_000))
feature_frame = pd.DataFrame(feature_matrix, columns=transformer.get_feature_names_out())
feature_frame.describe().T

In [ ]:
examples = clean.loc[
    clean["catalog_content"].str.contains("pack of| oz| ml| count", regex=True, na=False),
    ["sample_id", "catalog_content", "price"],
].head(20)
examples

In [ ]:
example_features = examples["catalog_content"].map(extract_catalog_features).tolist()
pd.concat(
    [examples.reset_index(drop=True), pd.DataFrame(example_features, columns=transformer.get_feature_names_out())],
    axis=1,
)

## Smoke Train

Run a sample first. After this works, run the full command from PowerShell for the real report.

In [ ]:
metadata = train_enhanced_baseline(sample_size=10_000, max_features=20_000)
metadata["metrics"]